In [4]:
from itertools import permutations

#Goal configurations
GOAL_R1 = tuple(range(1, 5))
GOAL_R2 = tuple(range(1, 9))
GOAL_R3 = tuple(range(1, 17))
BLANK = 16


#State space construction
def generate_row1_states():
    states = []
    for a in range(16):
        for b in range(16):
            for c in range(16):
                for d in range(16):
                    if len({a, b, c, d}) != 4:
                        continue
                    for blank in range(16):
                        if blank in {a, b, c, d}:
                            continue
                        BOARD = [0] * 16
                        BOARD[a], BOARD[b], BOARD[c], BOARD[d] = 1, 2, 3, 4
                        BOARD[blank] = BLANK
                        states.append(tuple(BOARD))
    return states

def generate_row2_states():
    states = []
    for a in range(4, 16):
        for b in range(4, 16):
            for c in range(4, 16):
                for d in range(4, 16):
                    if len({a, b, c, d}) != 4:
                        continue
                    for blank in range(4, 16):
                        if blank in {a, b, c, d}:
                            continue
                        BOARD = [1, 2, 3, 4] + [0] * 12
                        BOARD[a], BOARD[b], BOARD[c], BOARD[d] = 5, 6, 7, 8
                        BOARD[blank] = BLANK
                        states.append(tuple(BOARD))
    return states


def generate_row3_states():
    states = []
    for perm in permutations(range(8, 16)):
        BOARD = list(range(1, 9)) + [0] * 8
        for i in range(7):
            BOARD[perm[i]] = i + 9
        BOARD[perm[-1]] = BLANK
        states.append(tuple(BOARD))
    return states


row1_states = generate_row1_states()
row2_states = generate_row2_states()
row3_states = generate_row3_states()

In [5]:
#Transition helpers
def valid_swap(BOARD, blank, move):
    r, c = divmod(blank, 4)
    if move == 'l' and c > 0:
        return blank - 1
    if move == 'r' and c < 3:
        return blank + 1
    if move == 'u' and r > 0:
        return blank - 4
    if move == 'd' and r < 3:
        return blank + 4
    return None


def build_mdp(states, goal, active_len, base_penalty, goal_reward):
    mdp = {s: {} for s in states}
    actions = ['u', 'd', 'l', 'r']

    for state in states:
        blank = state.index(BLANK)
        prefix = state[:active_len]

        for act in actions:
            if prefix == goal:
                mdp[state][act] = [(1, state, 0, False)]
                continue

            BOARD = list(state)
            target = valid_swap(BOARD, blank, act)

            if target is None:
                mdp[state][act] = [(1, state, -67, False)]
                continue

            BOARD[blank], BOARD[target] = BOARD[target], BOARD[blank]

            correct = sum(BOARD[i] == goal[i] for i in range(active_len))
            reward = base_penalty + 0.5 * correct

            if tuple(BOARD[:active_len]) == goal:
                reward = goal_reward

            mdp[state][act] = [(1, tuple(BOARD), reward, False)]

    return mdp


mdp_r1 = build_mdp(row1_states, GOAL_R1, 4, -2, 67)
mdp_r2 = build_mdp(row2_states, GOAL_R2, 8, -5, 140)
mdp_r3 = build_mdp(row3_states, GOAL_R3, 16, -8, 250)

In [6]:
#Value Iteration
def value_iteration(states, goal, mdp, prefix_len, gamma=0.9, eps=1e-2):
    actions = ['u', 'd', 'l', 'r']
    V = {s: 0.0 for s in states}
    policy = {s: 'u' for s in states}

    for s in states:
        if s[:prefix_len] == goal:
            V[s] = 1.0

    while True:
        delta = 0
        for s in states:
            if s[:prefix_len] == goal:
                continue

            best_val = float('-inf')
            best_act = None

            for a in actions:
                p, ns, r, _ = mdp[s][a][0]
                val = p * (r + gamma * V.get(ns, 0))
                if val > best_val:
                    best_val = val
                    best_act = a

            delta = max(delta, abs(best_val - V[s]))
            V[s] = best_val
            policy[s] = best_act

        if delta < eps:
            break

    return policy


policy_r1 = value_iteration(row1_states, GOAL_R1, mdp_r1, 4)
policy_r2 = value_iteration(row2_states, GOAL_R2, mdp_r2, 8)
policy_r3 = value_iteration(row3_states, GOAL_R3, mdp_r3, 16)

In [1]:
#Execution helpers
def apply_move(state, action):
    BOARD = list(state)
    blank = BOARD.index(BLANK)
    target = valid_swap(BOARD, blank, action)
    if target is None:
        return state
    BOARD[blank], BOARD[target] = BOARD[target], BOARD[blank]
    return tuple(BOARD)


def mask_state(state, k):
    masked = list(state)

    # Zero out tiles beyond current phase
    for i in range(16):
        if masked[i] > k and masked[i] != BLANK:
            masked[i] = 0

    # Enforce frozen rows
    if k == 8:
        masked[0:4] = [1, 2, 3, 4]
    elif k == 16:
        masked[0:8] = [1, 2, 3, 4, 5, 6, 7, 8]

    return tuple(masked)


def print_board(state):
    for i in range(16):
        print(f"{state[i]:2}", end=" ")
        if i % 4 == 3:
            print()
    print()